# Join holiday calendar vào Dataset 1 và Dataset 2 đã có thời tiết

Notebook này join holiday calendar năm 2023 vào hai dataset đã được enrich weather:

- `data/dataset1/dataset1_with_weather.csv`
- `data/dataset2/dataset2_with_weather.csv`

Holiday được join theo `date`, sau đó tạo thêm:

- `is_holiday`
- `holiday_name`
- `holiday_type`
- `is_pre_holiday`
- `is_post_holiday`
- `days_to_nearest_holiday`

Output:

- `data/dataset1/dataset1_full.csv`
- `data/dataset2/dataset2_full.csv`

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)

if (Path.cwd() / 'data' / 'dataset1' / 'dataset1_with_weather.csv').exists():
    ROOT = Path.cwd()
elif (Path.cwd().parent / 'data' / 'dataset1' / 'dataset1_with_weather.csv').exists():
    ROOT = Path.cwd().parent
elif (Path.cwd().parent.parent / 'data' / 'dataset1' / 'dataset1_with_weather.csv').exists():
    ROOT = Path.cwd().parent.parent
else:
    ROOT = Path('.')

DATASET1_WEATHER_PATH = ROOT / 'data' / 'dataset1' / 'dataset1_with_weather.csv'
DATASET2_WEATHER_PATH = ROOT / 'data' / 'dataset2' / 'dataset2_with_weather.csv'
HOLIDAY_CALENDAR_PATH = ROOT / 'data' / 'holiday' / 'nyc_2023_holiday_calendar.csv'

DATASET1_OUTPUT_PATH = ROOT / 'data' / 'dataset1' / 'dataset1_full.csv'
DATASET2_OUTPUT_PATH = ROOT / 'data' / 'dataset2' / 'dataset2_full.csv'

DATASET1_WEATHER_PATH, DATASET2_WEATHER_PATH, HOLIDAY_CALENDAR_PATH

## 1. Đọc dữ liệu

In [ ]:
df1_weather = pd.read_csv(DATASET1_WEATHER_PATH, low_memory=False)
df2_weather = pd.read_csv(DATASET2_WEATHER_PATH, low_memory=False)
holiday_raw = pd.read_csv(HOLIDAY_CALENDAR_PATH)

print('Dataset 1 with weather shape:', df1_weather.shape)
print('Dataset 2 with weather shape:', df2_weather.shape)
print('Holiday calendar shape:', holiday_raw.shape)

display(df1_weather.head())
display(df2_weather.head())
display(holiday_raw)

## 2. Chuẩn hóa holiday calendar

In [ ]:
holiday = holiday_raw.copy()
holiday['date'] = pd.to_datetime(holiday['date'], errors='coerce').dt.normalize()

bool_cols = ['is_official_holiday', 'is_commercial_event']
for col in bool_cols:
    holiday[col] = holiday[col].astype(str).str.lower().map({'true': True, 'false': False})

# Nếu một ngày có nhiều event, gom lại để mỗi date chỉ có một dòng.
holiday = (
    holiday.groupby('date', as_index=False)
    .agg(
        holiday_name=('holiday_name', lambda s: ' | '.join(s.dropna().astype(str).unique())),
        holiday_type=('holiday_type', lambda s: ' | '.join(s.dropna().astype(str).unique())),
        is_official_holiday=('is_official_holiday', 'max'),
        is_commercial_event=('is_commercial_event', 'max'),
        notes=('notes', lambda s: ' | '.join(s.dropna().astype(str).unique())),
    )
)

# Trong project này, is_holiday nghĩa là ngày có trong holiday/event calendar.
# Các cột is_official_holiday và is_commercial_event giúp phân biệt lễ chính thức và event thương mại.
holiday['is_holiday'] = True

print('Holiday calendar after normalization:', holiday.shape)
print('Duplicate holiday dates:', holiday.duplicated('date').sum())
display(holiday)

## 3. Helper tạo feature trước/sau holiday

In [ ]:
holiday_dates = sorted(holiday['date'].dropna().unique())
holiday_date_set = set(holiday_dates)
holiday_name_by_date = dict(zip(holiday['date'], holiday['holiday_name']))

def nearest_holiday_info(date_value):
    if pd.isna(date_value):
        return pd.Series({'days_to_nearest_holiday': np.nan, 'nearest_holiday_name': np.nan})

    date_value = pd.Timestamp(date_value).normalize()
    distances = [(abs((date_value - pd.Timestamp(h_date)).days), pd.Timestamp(h_date)) for h_date in holiday_dates]
    min_distance, nearest_date = min(distances, key=lambda item: item[0])

    return pd.Series(
        {
            'days_to_nearest_holiday': int(min_distance),
            'nearest_holiday_name': holiday_name_by_date.get(nearest_date),
        }
    )

def add_holiday_features(df, dataset_name):
    out = df.copy()
    out['date'] = pd.to_datetime(out['date'], errors='coerce').dt.normalize()

    # Giữ holiday_name gốc của Dataset 1 để đối chiếu, không dùng nó làm holiday chính.
    if 'holiday_name' in out.columns:
        out = out.rename(columns={'holiday_name': 'original_holiday_name'})

    out = out.merge(
        holiday,
        on='date',
        how='left',
        validate='many_to_one',
    )

    out['is_holiday'] = out['is_holiday'].fillna(False).astype(bool)
    out['is_official_holiday'] = out['is_official_holiday'].fillna(False).astype(bool)
    out['is_commercial_event'] = out['is_commercial_event'].fillna(False).astype(bool)
    out['holiday_name'] = out['holiday_name'].fillna('No Holiday')
    out['holiday_type'] = out['holiday_type'].fillna('none')
    out['notes'] = out['notes'].fillna('')

    out['is_pre_holiday'] = out['date'].apply(lambda d: (pd.Timestamp(d).normalize() + pd.Timedelta(days=1)) in holiday_date_set)
    out['is_post_holiday'] = out['date'].apply(lambda d: (pd.Timestamp(d).normalize() - pd.Timedelta(days=1)) in holiday_date_set)

    nearest_info = out[['date']].drop_duplicates().copy()
    nearest_info = pd.concat([nearest_info, nearest_info['date'].apply(nearest_holiday_info)], axis=1)
    out = out.merge(nearest_info, on='date', how='left', validate='many_to_one')

    out['holiday_source'] = 'data/holiday/nyc_2023_holiday_calendar.csv'
    out['holiday_calendar_scope'] = 'NYC/US 2023 official holidays and selected commercial events'

    print(f'===== {dataset_name} holiday join summary =====')
    print('Rows:', len(out))
    print('Rows on holiday/event dates:', int(out['is_holiday'].sum()))
    print('Rows on official holidays:', int(out['is_official_holiday'].sum()))
    print('Rows on commercial events:', int(out['is_commercial_event'].sum()))
    print('Rows on pre-holiday dates:', int(out['is_pre_holiday'].sum()))
    print('Rows on post-holiday dates:', int(out['is_post_holiday'].sum()))
    print('\nTop holiday names by row count:')
    display(out.loc[out['is_holiday'], 'holiday_name'].value_counts())

    return out

## 4. Join holiday vào Dataset 1 và Dataset 2 đã có weather

In [ ]:
df1_full = add_holiday_features(df1_weather, 'Dataset 1')
df2_full = add_holiday_features(df2_weather, 'Dataset 2')

print('Dataset 1 full shape:', df1_full.shape)
print('Dataset 2 full shape:', df2_full.shape)

display(df1_full.head())
display(df2_full.head())

## 5. Kiểm tra chất lượng sau join

In [ ]:
holiday_cols = [
    'is_holiday',
    'holiday_name',
    'holiday_type',
    'is_official_holiday',
    'is_commercial_event',
    'is_pre_holiday',
    'is_post_holiday',
    'days_to_nearest_holiday',
    'nearest_holiday_name',
]

for name, df in [('Dataset 1', df1_full), ('Dataset 2', df2_full)]:
    print(f'===== {name} missing holiday fields =====')
    display(pd.DataFrame({'missing_count': df[holiday_cols].isna().sum(), 'missing_pct': df[holiday_cols].isna().mean() * 100}))
    print('\nHoliday rows by date:')
    display(
        df.loc[df['is_holiday']]
        .groupby(['date', 'holiday_name', 'holiday_type'], as_index=False)
        .size()
        .sort_values(['date'])
    )

if 'original_holiday_name' in df1_full.columns:
    print('Dataset 1 original_holiday_name comparison:')
    display(
        df1_full[['date', 'original_holiday_name', 'holiday_name', 'is_holiday']]
        .drop_duplicates()
        .sort_values('date')
        .head(40)
    )

## 6. Lưu file output

In [ ]:
DATASET1_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
DATASET2_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

df1_full.to_csv(DATASET1_OUTPUT_PATH, index=False, encoding='utf-8')
df2_full.to_csv(DATASET2_OUTPUT_PATH, index=False, encoding='utf-8')

print('Saved Dataset 1 full to:', DATASET1_OUTPUT_PATH)
print('Saved Dataset 2 full to:', DATASET2_OUTPUT_PATH)
print('Dataset 1 full shape:', df1_full.shape)
print('Dataset 2 full shape:', df2_full.shape)